In [1]:
"""
Orthohantavirus metadata cleaning & segmentation pipeline.
Revised for NCBI Datasets virus-genome TSV output.
"""

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional


def clean_orthohanta_metadata(
    input_tsv: str,
    output_tsv: str,
    dtype_hint: Optional[dict] = None
) -> pd.DataFrame:
    """
    Clean, segment, filter, remove empty columns,
    reorder columns by priority, and save output.
    """

    # STEP 1: Read TSV
    print("[1] Reading TSV...")
    df = pd.read_csv(
        input_tsv,
        sep="\t",
        low_memory=True
    )

    # Normalize column spacing
    df.columns = df.columns.str.strip()

    # Rename NCBI display headers to stable internal names
    df = df.rename(columns={
        "Accession": "accession",
        "BioProjects": "bioprojects",
        "BioSample accession": "biosample-acc",
        "Completeness": "completeness",
        "Gene count": "gene-count",
        "Length": "length",
        "Segment": "segment",
        "Release date": "release-date",
        "Update date": "update-date",
        "Source database": "sourcedb",
        "Geographic Location": "geo-location",
        "Geographic Region": "geo-region",
        "Geographic State": "geo-state",
        "Host Common Name": "host-common-name",
        "Host Name": "host-name",
        "Host Taxonomic ID": "host-tax-id",
        "Host Infraspecific Names Isolate": "host-infraspecific-isolate",
        "Host Infraspecific Names Strain": "host-infraspecific-strain",
        "Host Infraspecific Names Breed": "host-infraspecific-breed",
        "Host Infraspecific Names Cultivar": "host-infraspecific-cultivar",
        "Host Infraspecific Names Ecotype": "host-infraspecific-ecotype",
        "Host Infraspecific Names Sex": "host-infraspecific-sex",
        "Virus Common Name": "virus-common-name",
        "Virus Name": "virus-name",
        "Virus Taxonomic ID": "virus-tax-id",
        "Virus Infraspecific Names Isolate": "virus-infraspecific-isolate",
        "Virus Infraspecific Names Strain": "virus-infraspecific-strain",
        "Virus Infraspecific Names Breed": "virus-infraspecific-breed",
        "Virus Infraspecific Names Cultivar": "virus-infraspecific-cultivar",
        "Virus Infraspecific Names Ecotype": "virus-infraspecific-ecotype",
        "Virus Infraspecific Names Sex": "virus-infraspecific-sex",
        "Isolate Collection date": "isolate-collection-date",
        "Isolate Lineage": "isolate-lineage",
        "Isolate Lineage source": "isolate-lineage-source",
        "Purpose of Sampling": "purpose-of-sampling",
        "Submitter Country": "submitter-country",
        "Submitter country": "submitter-country",
        "Submitter Affiliation": "submitter-affiliation",
        "Submitter affiliation": "submitter-affiliation",
        "Submitter Names": "submitter-names",
        "Submitter names": "submitter-names",
        "SRA Accessions": "sra-accs",
        "SRA Accession(s)": "sra-accs",
        "Protein count": "protein-count",
        "Mature peptide count": "matpeptide-count",
        "Molecule type": "mol-type",
        "Is Annotated": "is-annotated",
        "Is Lab Host": "is-lab-host",
        "Is Vaccine Strain": "is-vaccine-strain",
        "Lab Host": "lab-host",
        "Virus Pangolin Classification": "virus-pangolin",
        "Host Pangolin Classification": "host-pangolin",
    })

    # Optional dtype coercion
    if dtype_hint is None:
        dtype_hint = {
            "accession": "string",
            "length": "Int64",
            "completeness": "string",
            "biosample-acc": "string",
            "virus-infraspecific-isolate": "string",
            "virus-infraspecific-strain": "string",
            "isolate-collection-date": "string",
            "segment": "string",
            "host-name": "string",
            "virus-name": "string",
            "geo-location": "string",
            "submitter-country": "string",
            "submitter-affiliation": "string",
            "submitter-names": "string",
        }

    for col, dtype in dtype_hint.items():
        if col in df.columns:
            try:
                df[col] = df[col].astype(dtype)
            except Exception:
                pass

    print(f"   Loaded {len(df):,} rows, {len(df.columns)} columns")

    required_cols = [
        "length",
        "completeness",
        "biosample-acc",
        "virus-infraspecific-isolate",
        "virus-infraspecific-strain",
        "isolate-collection-date",
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(
            f"Missing expected columns after renaming: {missing}\n"
            f"Available columns are: {df.columns.tolist()}"
        )

    # STEP 2: Segment classification
    print("[2] Classifying segments by length...")

    conditions = [
        (df["length"] >= 1000) & (df["length"] <= 3000),
        (df["length"] >= 3200) & (df["length"] <= 5000),
        (df["length"] >= 6500) & (df["length"] <= 12000),
    ]
    choices = ["S", "M", "L"]

    df["Calculated_Segment"] = np.select(
        conditions,
        choices,
        default="Partial/Unknown"
    )

    print(f"   Segment distribution:\n{df['Calculated_Segment'].value_counts(dropna=False)}")

    # STEP 3: Build group key
    print("[3] Building cascading group keys...")

    isolate = df["virus-infraspecific-isolate"].fillna("")
    strain = df["virus-infraspecific-strain"].fillna("")
    coll_date = df["isolate-collection-date"].fillna("")

    fallback_key = (
        isolate.astype(str) + "|" +
        strain.astype(str) + "|" +
        coll_date.astype(str)
    ).str.replace(r"\|+$", "", regex=True)

    df["Virus_Group_Key"] = df["biosample-acc"].where(
        df["biosample-acc"].notna() & (df["biosample-acc"].astype(str).str.strip() != ""),
        fallback_key
    )

    df["Virus_Group_Key"] = df["Virus_Group_Key"].replace("", pd.NA)

    print(f"   Created {df['Virus_Group_Key'].nunique(dropna=True):,} unique group keys")

    # STEP 4: Filter only groups with at least one COMPLETE row
    print("[4] Filtering groups by completeness...")

    df_filtered = df.dropna(subset=["Virus_Group_Key"]).groupby(
        "Virus_Group_Key",
        group_keys=False
    ).filter(lambda grp: (grp["completeness"] == "COMPLETE").any())

    print(f"   Before filter: {len(df):,} rows, {df['Virus_Group_Key'].nunique(dropna=True):,} groups")
    print(f"   After filter:  {len(df_filtered):,} rows, {df_filtered['Virus_Group_Key'].nunique(dropna=True):,} groups")

    # STEP 5: Drop fully empty columns
    print("[5] Dropping fully empty columns...")

    df_filtered = df_filtered.replace(r'^\s*$', np.nan, regex=True)
    df_filtered = df_filtered.dropna(axis=1, how='all')

    print(f"   Remaining columns after empty-column removal: {len(df_filtered.columns)}")

    # STEP 6: Reorder columns by priority
    print("[6] Reordering columns by priority...")

    priority_columns = [
        "accession",
        "virus-name",
        "virus-tax-id",
        "biosample-acc",
        "Virus_Group_Key",
        "Calculated_Segment",
        "segment",
        "length",
        "completeness",
        "gene-count",
        "protein-count",
        "matpeptide-count",
        "host-name",
        "host-tax-id",
        "geo-location",
        "geo-region",
        "geo-state",
        "bioprojects",
        "sra-accs",
        "sourcedb",
        "is-annotated",
        "is-lab-host",
        "lab-host",
        "isolate-lineage",
        "isolate-lineage-source",
        "release-date",
        "update-date",
        "isolate-collection-date",
        "submitter-country",
        "submitter-affiliation",
        "submitter-names",
    ]

    existing_priority = [col for col in priority_columns if col in df_filtered.columns]
    remaining_columns = [col for col in df_filtered.columns if col not in existing_priority]
    df_filtered = df_filtered[existing_priority + remaining_columns]

    # STEP 7: Sort rows
    print("[7] Sorting rows...")

    completeness_order = pd.CategoricalDtype(
        categories=["COMPLETE", "PARTIAL"],
        ordered=True
    )

    if "completeness" in df_filtered.columns:
        df_filtered["completeness"] = df_filtered["completeness"].astype(completeness_order)

    if "isolate-collection-date" in df_filtered.columns:
        df_filtered["isolate-collection-date-sort"] = pd.to_datetime(
            df_filtered["isolate-collection-date"],
            errors="coerce"
        )

    sort_cols = []
    ascending_flags = []

    if "completeness" in df_filtered.columns:
        sort_cols.append("completeness")
        ascending_flags.append(True)

    if "Calculated_Segment" in df_filtered.columns:
        sort_cols.append("Calculated_Segment")
        ascending_flags.append(True)

    if "isolate-collection-date-sort" in df_filtered.columns:
        sort_cols.append("isolate-collection-date-sort")
        ascending_flags.append(False)

    if "submitter-affiliation" in df_filtered.columns:
        sort_cols.append("submitter-affiliation")
        ascending_flags.append(True)

    if "submitter-names" in df_filtered.columns:
        sort_cols.append("submitter-names")
        ascending_flags.append(True)

    if sort_cols:
        df_filtered = df_filtered.sort_values(
            by=sort_cols,
            ascending=ascending_flags,
            na_position="last"
        )

    if "isolate-collection-date-sort" in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=["isolate-collection-date-sort"])

    # STEP 8: Save output
    print(f"[8] Writing output to {output_tsv}...")

    df_filtered.to_csv(
        output_tsv,
        sep="\t",
        index=False,
        na_rep="",
        lineterminator="\n"
    )

    print(f"   ✓ Saved {len(df_filtered):,} rows")
    print(f"   Final columns: {list(df_filtered.columns)}")

    return df_filtered


if __name__ == "__main__":
    INPUT_FILE = "data_report.tsv"
    OUTPUT_FILE = "orthohanta_cleaned.tsv"

    if Path(INPUT_FILE).exists():
        result_df = clean_orthohanta_metadata(INPUT_FILE, OUTPUT_FILE)
        print("\n✓ Pipeline complete!")
        print("\nFirst few rows of output:")
        print(result_df.head(10).to_string())
    else:
        print(f"⚠ Input file '{INPUT_FILE}' not found.")
        print("   Please provide a valid TSV file with required columns.")

[1] Reading TSV...
   Loaded 15,393 rows, 48 columns
[2] Classifying segments by length...
   Segment distribution:
Calculated_Segment
Partial/Unknown    9127
S                  3780
M                  1655
L                   831
Name: count, dtype: int64
[3] Building cascading group keys...
   Created 1,394 unique group keys
[4] Filtering groups by completeness...
   Before filter: 15,393 rows, 1,394 groups
   After filter:  6,768 rows, 180 groups
[5] Dropping fully empty columns...
   Remaining columns after empty-column removal: 31
[6] Reordering columns by priority...
[7] Sorting rows...
[8] Writing output to orthohanta_cleaned.tsv...
   ✓ Saved 6,768 rows
   Final columns: ['accession', 'virus-name', 'virus-tax-id', 'biosample-acc', 'Virus_Group_Key', 'Calculated_Segment', 'segment', 'length', 'completeness', 'gene-count', 'protein-count', 'matpeptide-count', 'host-name', 'host-tax-id', 'geo-location', 'geo-region', 'geo-state', 'bioprojects', 'sra-accs', 'sourcedb', 'is-annotate